# CalibraShift: When Confidence Lies
## Probability calibration under distribution shift

**Research question:** When a classifier assigns a high probability to its prediction, does that probability match observed reliability—and how does that relationship change when evaluation data shifts?

This exploratory project uses scikit-learn's built-in Breast Cancer Wisconsin Diagnostic dataset, so no large external dataset download is needed. This is a methods experiment, not a clinical tool.


## 1. Setup
Install the small set of packages used in this notebook.

In [ ]:
%pip -q install -U scikit-learn pandas numpy matplotlib seaborn


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import accuracy_score, balanced_accuracy_score, brier_score_loss, log_loss, roc_auc_score

RANDOM_STATE = 42
RESULTS = Path("results")
FIGURES = Path("figures")
RESULTS.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)
print("Setup complete.")


## 2. Load data and create a controlled shift
The target is encoded as malignant = 1 and benign = 0. The shifted test set is a controlled synthetic covariate shift: each test feature receives a modest offset equal to 0.35 training-set standard deviations. It is not intended to represent every real-world shift.

In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data.copy()
y = (data.target == 0).astype(int)  # 1=malignant, 0=benign

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
train_std = X_train.std(axis=0).replace(0, 1.0)
shift_strength = 0.35
X_test_shifted = X_test + shift_strength * train_std

print("Dataset shape:", X.shape)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print("Malignant prevalence in test set:", round(float(y_test.mean()), 3))
print("Synthetic shift strength:", shift_strength, "training standard deviations")


## 3. Metrics
Accuracy and balanced accuracy assess thresholded predictions. Brier score and log loss assess probability quality (lower is better). Expected Calibration Error (ECE) summarizes the gap between confidence and observed frequency across bins; it depends on binning. ROC-AUC assesses ranking.

In [ ]:
def expected_calibration_error(y_true, prob, n_bins=10):
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob, dtype=float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        if i == n_bins - 1:
            mask = (prob >= edges[i]) & (prob <= edges[i + 1])
        else:
            mask = (prob >= edges[i]) & (prob < edges[i + 1])
        if mask.any():
            ece += mask.mean() * abs(y_true[mask].mean() - prob[mask].mean())
    return float(ece)

def evaluate(y_true, prob, model_name, condition, calibration):
    pred = (prob >= 0.5).astype(int)
    return {
        "model": model_name, "test_condition": condition, "calibration": calibration,
        "n_test": len(y_true),
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "roc_auc": roc_auc_score(y_true, prob),
        "brier_score": brier_score_loss(y_true, prob),
        "log_loss": log_loss(y_true, np.clip(prob, 1e-7, 1 - 1e-7)),
        "ece_10_bins": expected_calibration_error(y_true, prob, n_bins=10),
    }


## 4. Train models and fit calibration
Calibration is fitted using cross-validation on training data only. The test sets are not used to fit the models or calibrators.

In [ ]:
base_models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                                         random_state=RANDOM_STATE))
    ])
}

fitted = {}
rows = []
probabilities = {}

for name, estimator in base_models.items():
    estimator.fit(X_train, y_train)
    calibrated = CalibratedClassifierCV(estimator=estimator, method="sigmoid", cv=5)
    calibrated.fit(X_train, y_train)
    for calibration_name, model in [
        ("Uncalibrated", estimator),
        ("Sigmoid calibrated", calibrated),
    ]:
        for condition, X_eval in [
            ("In-distribution", X_test),
            ("Synthetic feature shift", X_test_shifted),
        ]:
            prob = model.predict_proba(X_eval)[:, 1]
            probabilities[(name, calibration_name, condition)] = prob
            rows.append(evaluate(y_test, prob, name, condition, calibration_name))

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(RESULTS / "calibration_shift_metrics.csv", index=False)
display(metrics_df.round(4))


## 5. Reliability diagrams
Points close to the diagonal indicate that predicted probabilities align with observed positive frequencies.

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)
for ax, condition in zip(axes, ["In-distribution", "Synthetic feature shift"]):
    ax.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
    for name in base_models:
        for calibration_name, linestyle in [("Uncalibrated", "-"), ("Sigmoid calibrated", ":")]:
            prob = probabilities[(name, calibration_name, condition)]
            frac_pos, mean_pred = calibration_curve(y_test, prob, n_bins=8, strategy="quantile")
            ax.plot(mean_pred, frac_pos, marker="o", linestyle=linestyle,
                    label=f"{name} — {calibration_name}")
    ax.set_title(condition)
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Observed positive fraction")
    ax.legend(fontsize=7)
fig.suptitle("Reliability diagrams: calibration under distribution shift")
fig.tight_layout()
fig.savefig(FIGURES / "reliability_diagrams.png", dpi=180, bbox_inches="tight")
plt.show()


## 6. Compare probability quality
Lower Brier score and ECE indicate better probability quality, though ECE depends on binning. Calibration does not guarantee improved accuracy or robustness.

In [ ]:
plot_df = metrics_df.copy()
plot_df["series"] = plot_df["model"] + "\n" + plot_df["calibration"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=plot_df, x="series", y="brier_score", hue="test_condition", ax=axes[0])
axes[0].set_title("Brier score (lower is better)")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=20)
sns.barplot(data=plot_df, x="series", y="ece_10_bins", hue="test_condition", ax=axes[1])
axes[1].set_title("ECE, 10 bins (lower is better)")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=20)
fig.tight_layout()
fig.savefig(FIGURES / "probability_quality_under_shift.png", dpi=180, bbox_inches="tight")
plt.show()


## 7. Save summary outputs

In [ ]:
best_rows = metrics_df.sort_values("brier_score").groupby(
    ["model", "test_condition"], as_index=False
).first()
best_rows.to_csv(RESULTS / "best_brier_by_condition.csv", index=False)

config = {
    "dataset": "Breast Cancer Wisconsin Diagnostic (scikit-learn built-in)",
    "random_state": RANDOM_STATE,
    "n_total": int(len(X)), "n_train": int(len(X_train)), "n_test": int(len(X_test)),
    "shift_strength_std_units": shift_strength,
    "models": list(base_models.keys()),
    "calibration_methods": ["Uncalibrated", "Sigmoid calibrated"],
    "metrics": ["accuracy", "balanced_accuracy", "roc_auc", "brier_score", "log_loss", "ece_10_bins"],
    "note": "Synthetic test-only feature shift; exploratory results, not a clinical evaluation."
}
(RESULTS / "experiment_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
display(best_rows.round(4))


## 8. Limitations
- The shift is synthetic and simplified; it does not represent every real-world distribution shift.
- A single train/test split is subject to sampling variability.
- ECE depends on the binning strategy.
- The dataset is medical in subject matter but this notebook is not for diagnosis or clinical deployment.
- Calibration and model selection must not use the test set.


## 9. Verify generated files

In [ ]:
expected = [
    RESULTS / "calibration_shift_metrics.csv",
    RESULTS / "best_brier_by_condition.csv",
    RESULTS / "experiment_config.json",
    FIGURES / "reliability_diagrams.png",
    FIGURES / "probability_quality_under_shift.png",
]
verification = pd.DataFrame({"file": [str(p) for p in expected],
                              "exists": [p.exists() for p in expected]})
display(verification)
assert verification["exists"].all(), "Some outputs are missing. Run cells above in order."
print("All expected output files exist.")


## 10. Download results and figures as a ZIP (Google Colab)
Run after the preceding cells finish.

In [ ]:
from google.colab import files
import zipfile

zip_path = Path("CalibraShift_results_and_figures.zip")
files_to_zip = [
    RESULTS / "calibration_shift_metrics.csv",
    RESULTS / "best_brier_by_condition.csv",
    RESULTS / "experiment_config.json",
    FIGURES / "reliability_diagrams.png",
    FIGURES / "probability_quality_under_shift.png",
]
missing = [str(p) for p in files_to_zip if not p.exists()]
if missing:
    raise FileNotFoundError("Missing outputs: " + ", ".join(missing))
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in files_to_zip:
        zf.write(p, arcname=str(p))
print(f"Created {zip_path} ({zip_path.stat().st_size:,} bytes)")
files.download(str(zip_path))
